In [ ]:
import torch.nn as nn
import torch

In [ ]:
# get the data set of 32x32 RGB images from the CIFAR-10 dataset
from torchvision import datasets

# define where the data will be stored
data_path = '../../data/data-unversioned/p1ch7/'

# download the training data
cifar10 = datasets.CIFAR10(data_path, train = True, download = True)

# download the validation data
cifar10_val = datasets.CIFAR10(data_path, train = False, download = True)

In [ ]:
from torchvision import transforms

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

cifar10 = datasets.CIFAR10(
    data_path, train=True, download=False,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4915, 0.4823, 0.4468),
                             (0.2470, 0.2435, 0.2616))
    ]))
cifar10_val = datasets.CIFAR10(
    data_path, train=False, download=False,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4915, 0.4823, 0.4468),
                             (0.2470, 0.2435, 0.2616))
    ]))

In [ ]:
# now we are going to subset the data
# to just consider birds and planes
label_map = {0:0, 2:1}
class_names = ['airplane','bird']
cifar2 = [(img, label_map[label]) for img, label in cifar10 if label in [0,2]]
cifar2_val = [(img, label_map[label]) for img, label in cifar10_val if label in [0,2]]

In [ ]:
model = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1), # 3 RGB channels
    nn.Tanh(),
    nn.MaxPool2d(2), # note that we have 16 features, but each feature is still a NxN "image"
    nn.Conv2d(16, 8, kernel_size=3, padding=1), # MaxPooling would have made the feature ("image") have the size
    nn.Tanh(),
    nn.MaxPool2d(2),
    nn.Flatten(), # not best practice, apparently
    nn.Linear(8 * 8 * 8, 32), # converted the features of shape 8x8 into vectors and output a 32 dim vector
    nn.Tanh(),
    nn.Linear(32,2) # finally obtain the 2 output featurs for the one-hot-encoding
)

In [ ]:
img, _ = cifar2[0]
model(img.unsqueeze(0))

In [ ]:
# sub-classing nn.Module

class Net(nn.Module):
    def __init__(self):
        super().__init__() # required to obtain instantiated attributes
        # all submodules (transformations with parameters) should be defined here
        # regardless of being custom made or predefined via PyTorch
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.act1 = nn.Tanh()
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 8, kernel_size=3, padding=1)
        self.act2 = nn.Tanh()
        self.pool2 = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(8*8*8, 32)
        self.act3 = nn.Tanh()
        self.fc2 = nn.Linear(32,2)

    def forward(self, x): # method needed for all nn.Module subclasses
        # first convolutional module
        out = self.pool1(self.act1(self.conv1(x)))
        # second convolutional module
        out = self.pool2(self.act2(self.conv2(out)))
        # transformation of the data between modules
        out = out.view(-1,8*8*8) # the -1 as the batch dimension is to ensure whatever dimension is left over is the batch dimension
        # first fully connected module
        out = self.act3(self.fc1(out))
        # final fully connected module
        out = self.fc2(out)
        return out

In [ ]:
model = Net()

numel_list = [p.numel() for p in model.parameters()]
sum(numel_list), numel_list

Each nn.Module class is an object, and hence it has some internal "state". In the context of PyTorch, the state is a collection of parameters. Each nn.Module class has a functional equivalent in nn.functional. The idea here is that the function has no internal state, the "state" is an input. In other words, we need to put in the parameters along with the typical arguments for the module.

For example, nn.Linear can take in an input vector, but its instantiation will generate a random set of parameters, i.e. a random state. When we call nn.Linear(x), the transformation will occur on x with the instantiated state. However, we can also run 
    
    nn.functional.linear(x, weight, bias)

if we know the weight matrix and bias we would like to use. Note the uncapitalized naming for the non-object.

One way to use this is to not both instantiating transformations like Tanh() that do not use parameters, and so have an "empty state".

Let us rewrite the Net class using this knowledge to tighten up the code a bit.

I would imagine we could get creative an set an input for the constructor that would be the activation function we wish to use (e.g. ReLU,Tanh, etc). This way, we could instantiate a module with slightly different modules very easily.

Note that for something like quantization, the stateless functions actually get a state. So, if we play to quantize a model we may want to explicitly write out the functions a modules for those later purposes.

In [ ]:
# sub-classing nn.Module

import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super().__init__() # required to obtain instantiated attributes
        # all submodules (transformations with parameters) should be defined here
        # regardless of being custom made or predefined via PyTorch
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 8, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(8*8*8, 32)
        self.fc2 = nn.Linear(32,2)

    def forward(self, x): # method needed for all nn.Module subclasses
        # first convolutional module
        out = F.max_pool2d(torch.tanh(self.conv1(x)),2) # note that we could also use F.tanh, but this is deprecated
        # second convolutional module
        out = F.max_pool2d(torch.tanh(self.conv2(out)),2)
        # transformation of the data between modules
        out = out.view(-1,8*8*8) # the -1 as the batch dimension is to ensure whatever dimension is left over is the batch dimension
        # first fully connected module
        out = torch.tanh(self.fc1(out))
        # final fully connected module
        out = self.fc2(out)
        return out

In [ ]:
model = Net()

model(img.unsqueeze(0))

In [ ]:
# define the training loop

import datetime

def training_loop(n_epochs, optimizer, model, loss_fn, train_loader):
    for epoch in range(1,n_epochs + 1):
        loss_train = 0.0
        for imgs, labels in train_loader:
            outputs = model(imgs)
            loss = loss_fn(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_train += loss.item()
    if epoch == 1 or epoch % 10 == 0:
        print("{} Epoch {}, Training loss {}".format(
            datetime.datetime.now(), epoch, 
            loss_train / len(train_loader)
        ))

In [ ]:
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64, shuffle=True)

model = Net()
optimizer = torch.optim.SGD(model.parameters(),lr=1e-2)
loss_fn = torch.nn.CrossEntropyLoss()

training_loop(
    n_epochs = 100,
    optimizer = optimizer,
    model = model,
    loss_fn = loss_fn,
    train_loader = train_loader
)

The book, published circa 2019 remarked that depending on hardware, the training could take up to 20 minutes. On this M1 Macbook Pro, it took 4 minutes! The training loss if comparable to what was obtained in the book (it's actually better), so I am going to assume that SGD did its job well enough.

In [ ]:
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=65, shuffle=False)
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=65, shuffle=False)

def validate(model, train_loader, val_loader):
    for name, loader in [("train", train_loader),("val", val_loader)]:
        correct = 0
        total = 0

        with torch.no_grad():
            for imgs, labels in loader:
                outputs = model(imgs)
                _, predicted = torch.max(outputs, dim=1) # we don't care about the index
                total += labels.shape[0]
                correct += int((predicted == labels).sum())
        print("Accuracy {}: {:.2f}".format(name, correct/total))

validate(model, train_loader, val_loader)

The fully connected network achieved an accuracy of 75% based on the book's code. I could go back and see if a different architecture would yield better results. But either way, this architecture achieved a significant improvement. 

Something to add to a validate function would be true/false-positive/negative rates, especially broken down by label. This would give a more honest picture of the performance and would, for real world applications, but more useful. 

In [ ]:
# save the model
torch.save(model.state_dict(), data_path + 'birds_vs_airplanes.pt')

In [ ]:
# when loading a model, we would need the model architecture along with the weights and biases. This would be 

loaded_model = Net() # imported from some .py file where the model architeture is actually written
loaded_model.load_state_dict(torch.load(data_path + 'birds_vs_airplanes.pt')) 

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")


In [ ]:
print(F"training on {device}")

In [ ]:
# retrain on the GPU

def training_loop(n_epochs, optimizer, model, loss_fn, train_loader):
    for epoch in range(1,n_epochs + 1):
        loss_train = 0.0
        for imgs, labels in train_loader:
            # send data to the device
            imgs = imgs.to(device = device)
            labels = labels.to(device = device)
            outputs = model(imgs)
            loss = loss_fn(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_train += loss.item()
    if epoch == 1 or epoch % 10 == 0:
        print("{} Epoch {}, Training loss {}".format(
            datetime.datetime.now(), epoch, 
            loss_train / len(train_loader)
        ))

In [ ]:
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64, shuffle=True)

model = Net().to(device=device)
optimizer = torch.optim.SGD(model.parameters(),lr=1e-2) #implicitly sent to same device as model (parameters)
loss_fn = torch.nn.CrossEntropyLoss()

training_loop(
    n_epochs = 100,
    optimizer = optimizer,
    model = model,
    loss_fn = loss_fn,
    train_loader = train_loader
)

We get comparable accuracy on the training set but at 1/4 of the total time (1 minute)! This means we could easily play with the hyperparameters to check various optimizations.

In [ ]:
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64, shuffle=False)
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=64, shuffle=False)

# redefine the validation function to handle device locations
def validate(model, train_loader, val_loader):
    for name, loader in [("train", train_loader),("val", val_loader)]:
        correct = 0
        total = 0

        with torch.no_grad():
            for imgs, labels in loader:
                imgs = imgs.to(device=device)
                labels = labels.to(device=device)
                outputs = model(imgs)
                _, predicted = torch.max(outputs, dim=1) # we don't care about the index
                total += labels.shape[0]
                correct += int((predicted == labels).sum())
        print("Accuracy {}: {:.2f}".format(name, correct/total))

validate(model, train_loader, val_loader)

Marginally worse performance on the validation set compared to the CPU run, but that can be purely explained by the stochastic nature of the optimizer.

In [ ]:
# do the default save/load on current device, we can override as follows
# either send to CPU and then save, or save and then load to the desired device, regardless of location.
# I will do the former and the latter in order to avoid issues where the device doesn't exist in the first place

saved_model = model.to(device='cpu')
torch.save(saved_model.state_dict(), data_path + 'birds_vs_airplanes.pt')

In [ ]:
loaded_model = Net().to(device=device)
loaded_model.load_state_dict(torch.load(data_path + 'birds_vs_airplanes.pt', map_location=device))